# Notebook 03: V-JEPA 2-AC — Action-Conditioned World Model for Robotics

**Goal:** Understand how V-JEPA 2 becomes a robot world model by conditioning on actions.

This is the bridge from self-supervised video understanding → robot control.

---

## Key Idea

V-JEPA 2 learns: *"Given part of a video, predict representations of the rest"*

V-JEPA 2-AC learns: *"Given the current state + an action, predict the next state representation"*

```
V-JEPA 2 (pretraining):   visible_patches → predictor → masked_patch_representations
V-JEPA 2-AC (robotics):   frame_t + action_t + state_t → predictor → frame_{t+1} representation
```

## Architecture

```
Frame 0: [z₀₁, z₀₂, ..., z₀₂₅₆]  ← 256 spatial tokens from frozen encoder
Frame 1: [z₁₁, z₁₂, ..., z₁₂₅₆]
  ...
Frame 7: [z₇₁, z₇₂, ..., z₇₂₅₆]

For each frame, INTERLEAVE conditioning tokens:
Frame k: [action_k, state_k, z_k1, z_k2, ..., z_k256]
           ↑ 7-dim    ↑ 7-dim    ↑ 256 spatial patch tokens

Apply FRAME-CAUSAL attention:
  Frame 0 can see: Frame 0 only
  Frame 1 can see: Frame 0, Frame 1
  Frame k can see: Frame 0, ..., Frame k  (NOT future frames)

Output: predicted next-frame representations
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import math

torch.manual_seed(42)

## 0.5 The 5W+H of V-JEPA 2-AC (Action-Conditioned World Model)

### WHO developed V-JEPA 2-AC?
**Meta FAIR** — same team as V-JEPA 2. The robotics extension was developed in collaboration with the Embodied AI team at Meta, using the **DROID dataset** (62 hours of diverse robot manipulation).

### WHAT is V-JEPA 2-AC?
A **latent world model** that predicts future visual representations conditioned on robot actions. Given the current visual state $z_t$ and an action $a_t$, it predicts the next state $\hat{z}_{t+1}$ — all in latent space, not pixels.

### WHERE does V-JEPA 2-AC sit in the robotics stack?

| Level | What it does | V-JEPA 2-AC's role |
|-------|-------------|-------------------|
| **Perception** | See the world | Frozen encoder extracts $z_t$ from camera images |
| **World model** | Predict consequences of actions | AC predictor: $\hat{z}_{t+1} = P(a_t, s_t, z_t)$ |
| **Planning** | Choose actions to reach goals | CEM searches for best action sequence in latent space |
| **Control** | Execute on hardware | First planned action sent to robot |

### WHEN is each component active?

| Phase | Frozen Encoder | AC Predictor | CEM Planner |
|-------|---------------|--------------|-------------|
| **Post-training** | Generates targets | Being trained | Not used |
| **Inference** | Encodes current frame | Evaluates action candidates | Searches for optimal actions |

### WHY predict in latent space for robotics?

**Speed argument:** A 256×256 RGB image has 196,608 values. A V-JEPA 2 latent frame has 256 × 1408 = 360,448 values — comparable, but:
- Latent states are **semantically structured** (not raw pixels)
- The AC predictor operates on **1024-dim** predictor space → much smaller
- CEM evaluates 800 candidates → latent prediction takes ~20ms each, pixel prediction takes ~300ms each
- Result: **16 sec** (latent) vs **~4 min** (pixel, Cosmos) for planning

**Quality argument:** Pixel-space models waste capacity predicting exact textures/lighting. V-JEPA 2-AC predicts **what matters** — object positions, gripper state, contact events.

### HOW does the action conditioning work? (Matrix-level)

The key insight: actions and states are **interleaved** as extra tokens in the sequence.

$$\text{Per frame } k: \quad \underbrace{[\mathbf{a}_k}_{\in \mathbb{R}^{1 \times D_p}}, \underbrace{\mathbf{s}_k}_{\in \mathbb{R}^{1 \times D_p}}, \underbrace{\mathbf{z}_{k,1}, ..., \mathbf{z}_{k,HW}}_{\in \mathbb{R}^{HW \times D_p}}]$$

where $\mathbf{a}_k = W_a \cdot a_k^{\text{raw}} \in \mathbb{R}^{D_p}$ (linear projection from 7→1024) and $\mathbf{s}_k = W_s \cdot s_k^{\text{raw}} \in \mathbb{R}^{D_p}$.

The full sequence for $T$ frames:
$$\mathbf{X} = [\mathbf{a}_0, \mathbf{s}_0, \mathbf{z}_{0,:}, \mathbf{a}_1, \mathbf{s}_1, \mathbf{z}_{1,:}, ..., \mathbf{a}_{T-1}, \mathbf{s}_{T-1}, \mathbf{z}_{T-1,:}] \in \mathbb{R}^{T(2+HW) \times D_p}$$

Frame-causal attention ensures frame $k$ only sees frames $0, ..., k$:
$$\text{Attention}(\mathbf{Q}_k, \mathbf{K}_{0:k}, \mathbf{V}_{0:k}) \quad \text{(causal mask blocks future frames)}$$

## 1. Action & State Representation

From `refs/vjepa2/app/vjepa_droid/droid.py`:

- **End-effector state** $s_k \in \mathbb{R}^7$: 3D position + 3D orientation (Euler angles) + gripper state
- **Action** $a_k = s_{k+1} - s_k \in \mathbb{R}^7$: relative end-effector displacement
- **Camera extrinsics** (optional): 6D camera pose

These are encoded via simple linear projections:
```python
# From refs/vjepa2/src/models/ac_predictor.py, lines 53-56
self.action_encoder = nn.Linear(7, predictor_embed_dim)   # 7 → 1024
self.state_encoder = nn.Linear(7, predictor_embed_dim)    # 7 → 1024
self.extrinsics_encoder = nn.Linear(6, predictor_embed_dim)  # 6 → 1024
```

In [ ]:
# Simulate robot trajectory data
B = 2          # batch size
T = 8          # 8 frames at 4 FPS = 2 seconds
H, W = 16, 16  # spatial grid (256px / 16 patch_size)
D_enc = 1408   # ViT-giant encoder dim
D_pred = 1024  # AC predictor dim
action_dim = 7

# Simulated frozen encoder outputs (one per frame)
frame_features = torch.randn(B, T, H * W, D_enc)  # [2, 8, 256, 1408]

# Robot actions: delta end-effector state
actions = torch.randn(B, T - 1, action_dim) * 0.01  # [2, 7, 7] — small movements

# Robot states: absolute end-effector pose
states = torch.randn(B, T, action_dim)  # [2, 8, 7]

print(f"Frame features: {frame_features.shape}  — from frozen ViT-g encoder")
print(f"Actions:        {actions.shape}  — delta EE state (T-1 because last frame has no action)")
print(f"States:         {states.shape}  — absolute EE pose")
print(f"")
print(f"Each frame has {H*W} = {H}×{W} spatial tokens of dim {D_enc}")
print(f"Total tokens per frame (with action+state): {H*W + 2} = {H*W} spatial + 1 action + 1 state")

## 2. Frame-Causal Attention Mask

This is critical for autoregressive prediction. From `refs/vjepa2/src/models/utils/modules.py`:

```python
def build_action_block_causal_attention_mask(T, H, W, add_tokens=2):
    N_T = add_tokens + (H * W)   # tokens per frame
    N = T * N_T                   # total sequence length
    mask = torch.zeros(N, N).bool()
    mask_block = torch.ones(N_T, N_T).bool()
    for t1 in range(T):
        for t2 in range(0, t1 + 1):       # can attend to past + current
            mask[t1*N_T:(t1+1)*N_T, t2*N_T:(t2+1)*N_T] = mask_block
    return mask
```

In [ ]:
def build_action_block_causal_attention_mask(T, H, W, add_tokens=2):
    """Build frame-causal attention mask.
    
    Each frame's tokens (action + state + patches) can attend to
    all tokens from the SAME frame and ALL PREVIOUS frames,
    but NOT future frames.
    
    Source: refs/vjepa2/src/models/utils/modules.py, lines 12-23
    """
    N_T = add_tokens + (H * W)  # tokens per frame
    N = T * N_T                  # total sequence length
    mask = torch.zeros(N, N).bool()
    mask_block = torch.ones(N_T, N_T).bool()
    
    for t1 in range(T):
        for t2 in range(0, t1 + 1):  # causal: past + current only
            mask[t1 * N_T : (t1 + 1) * N_T, t2 * N_T : (t2 + 1) * N_T] = mask_block
    
    return mask

# Visualize with small dimensions
T_vis, H_vis, W_vis = 4, 2, 2  # 4 frames, 2x2 spatial grid
mask = build_action_block_causal_attention_mask(T_vis, H_vis, W_vis, add_tokens=2)

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(mask.float().numpy(), cmap='Blues', vmin=0, vmax=1)

# Add frame boundaries
N_T = 2 + H_vis * W_vis  # 6 tokens per frame
for t in range(1, T_vis):
    ax.axhline(y=t * N_T - 0.5, color='red', linewidth=2)
    ax.axvline(x=t * N_T - 0.5, color='red', linewidth=2)

ax.set_xlabel('Key position (can attend TO)')
ax.set_ylabel('Query position (attending FROM)')
ax.set_title(f'Frame-Causal Attention Mask ({T_vis} frames, {N_T} tokens/frame)\n'
             f'Blue = can attend, White = blocked\n'
             f'Red lines = frame boundaries')
plt.tight_layout()
plt.show()

print(f"Tokens per frame: {N_T} (2 conditioning + {H_vis*W_vis} spatial)")
print(f"Frame 0 sees: only itself")
print(f"Frame 1 sees: frames 0-1")
print(f"Frame 3 sees: frames 0-3 (all past)")
print(f"")
print(f"Real V-JEPA 2-AC: {8} frames × ({2} + {16*16}) = {8 * (2 + 256)} total tokens")

## 3. Token Interleaving: How Actions Enter the Sequence

From `refs/vjepa2/src/models/ac_predictor.py`, lines 136-153:

```python
def forward(self, x, actions, states, extrinsics=None):
    x = self.predictor_embed(x)              # project to predictor dim
    s = self.state_encoder(states).unsqueeze(2)   # [B, T, 1, D]
    a = self.action_encoder(actions).unsqueeze(2)  # [B, T, 1, D]
    x = x.view(B, T, H*W, D)                # [B, T, 256, D]
    x = torch.cat([a, s, x], dim=2)          # [B, T, 2+256, D]
    x = x.flatten(1, 2)                      # [B, T*(2+256), D]
```

Per frame the token order is: **[action, state, patch₁, patch₂, ..., patch₂₅₆]**

In [ ]:
def demonstrate_token_interleaving():
    """Show how V-JEPA 2-AC interleaves action/state tokens with spatial tokens."""
    B, T, HW, D = 1, 4, 4, 8  # tiny dims for visualization
    action_dim = 7
    
    # Simulated inputs
    x = torch.randn(B, T * HW, D)  # flattened frame features
    actions = torch.randn(B, T, action_dim)
    states = torch.randn(B, T, action_dim)
    
    # Encode actions and states
    action_enc = nn.Linear(action_dim, D)
    state_enc = nn.Linear(action_dim, D)
    
    a = action_enc(actions).unsqueeze(2)   # [B, T, 1, D]
    s = state_enc(states).unsqueeze(2)     # [B, T, 1, D]
    x_reshaped = x.view(B, T, HW, D)      # [B, T, HW, D]
    
    # Interleave: [action, state, patch1, ..., patchHW] per frame
    x_interleaved = torch.cat([a, s, x_reshaped], dim=2)  # [B, T, 2+HW, D]
    x_flat = x_interleaved.flatten(1, 2)                   # [B, T*(2+HW), D]
    
    print(f"Before interleaving:")
    print(f"  Frame features: {x.shape}  (B, T*HW, D)")
    print(f"  Actions:        {actions.shape}")
    print(f"  States:         {states.shape}")
    print(f"")
    print(f"After interleaving:")
    print(f"  Shape: {x_flat.shape}  (B, T*(2+HW), D)")
    print(f"")
    print(f"Token layout per frame (total {2+HW} tokens):")
    for t in range(T):
        start = t * (2 + HW)
        print(f"  Frame {t}: [action_{t}, state_{t}, patch_{t}_0, patch_{t}_1, patch_{t}_2, patch_{t}_3]")
        print(f"           positions [{start}..{start + 2 + HW - 1}]")

demonstrate_token_interleaving()

## 4. ACRoPEAttention: Different RoPE for Action vs Spatial Tokens

From `refs/vjepa2/src/models/utils/modules.py`, ACRoPEAttention (lines 114-263):

**Key insight:** Action/state tokens don't have spatial positions — they only have temporal positions.

```
Spatial tokens: get full 3D RoPE (depth + height + width)
Action tokens:  get only 1D RoPE (depth/temporal only)
```

The implementation:
1. Split out action tokens from the sequence
2. Compute Q,K,V separately for action and spatial tokens
3. Apply temporal-only RoPE to action Q,K
4. Apply full 3D RoPE to spatial Q,K
5. Merge back together for joint attention

In [ ]:
# Demonstrate the split RoPE logic
T, H, W = 4, 4, 4
action_tokens = 2  # action + state
tokens_per_frame = action_tokens + H * W  # 18

print("ACRoPEAttention: How action vs spatial tokens get different position encodings")
print("=" * 70)
print(f"")
print(f"Tokens per frame: {tokens_per_frame} = {action_tokens} conditioning + {H*W} spatial")
print(f"")

for t in range(min(T, 2)):  # show first 2 frames
    print(f"Frame {t}:")
    print(f"  Action token [idx {t*tokens_per_frame}]:")
    print(f"    → Temporal RoPE only: depth={t}, height=N/A, width=N/A")
    print(f"  State token  [idx {t*tokens_per_frame+1}]:")
    print(f"    → Temporal RoPE only: depth={t}, height=N/A, width=N/A")
    
    # Show a few spatial tokens
    for p in range(min(3, H*W)):
        flat_idx = p
        h_pos = flat_idx // W
        w_pos = flat_idx % W
        print(f"  Patch [{h_pos},{w_pos}] [idx {t*tokens_per_frame+action_tokens+p}]:")
        print(f"    → Full 3D RoPE: depth={t}, height={h_pos}, width={w_pos}")
    print(f"  ...")
    print()

### 5.5 Detailed Math: Teacher-Forcing vs Autoregressive

**Teacher-Forcing (TF):**

At each step $k$, use **ground-truth** encoded frames from the target encoder:

$$\hat{z}_{k+1} = P_\phi(\{a_t, s_t, \underbrace{E_{\bar\theta}(x_t)}_{\text{ground truth}}\}_{t=0}^{k})$$

$$\mathcal{L}_{\text{tf}} = \frac{1}{T-1} \sum_{k=1}^{T-1} \frac{1}{HW \cdot D} \sum_{n,d} |\hat{z}_{k}^{(n,d)} - z_{k}^{(n,d)}|$$

where $z_k = \text{LayerNorm}(E_{\bar\theta}(x_k))$ is the target.

**Autoregressive Rollout (AR):**

Only frame 0 uses ground truth. All subsequent frames use the **predictor's own outputs**:

$$\hat{z}_1 = P_\phi(a_0, s_0, z_0) \quad \text{(z₀ is GT)}$$
$$\hat{z}_2 = P_\phi(a_{0:1}, s_{0:1}, [z_0, \hat{z}_1]) \quad \text{(ẑ₁ is predicted!)}$$
$$\hat{z}_3 = P_\phi(a_{0:2}, s_{0:2}, [z_0, \hat{z}_1, \hat{z}_2]) \quad \text{(ẑ₁, ẑ₂ are predicted!)}$$

$$\mathcal{L}_{\text{ar}} = \frac{1}{\text{auto\_steps}} \sum_{k=1}^{\text{auto\_steps}} \frac{1}{HW \cdot D} \sum_{n,d} |\hat{z}_{k}^{(n,d)} - z_{k}^{(n,d)}|$$

**Why both losses?**
- TF alone → model relies on perfect inputs → fails when using own predictions (error accumulation)
- AR alone → hard to train (errors compound from step 1) → slow convergence
- **Both together:** TF provides stable gradient signal, AR teaches self-correction
- `auto_steps` parameter controls how many AR steps (typically 2-3)

## 5. Training Losses: Teacher-Forcing + Autoregressive Rollout

V-JEPA 2-AC has **two loss components** (from `refs/vjepa2/app/vjepa_droid/train.py`):

### Loss 1: Teacher-Forcing
$$\mathcal{L}_{\text{tf}}(\phi) = \frac{1}{T} \sum_{k=1}^{T} \| P_\phi((a_t, s_t, E(x_t))_{t \leq k}) - E(x_{k+1}) \|_1$$

At each step, the predictor gets **ground-truth** encoded frames as input.

### Loss 2: Autoregressive Rollout
$$\mathcal{L}_{\text{ar}}(\phi) = \| P_\phi(a_{1:T}; s_1, z_1) - z_{T+1} \|_1$$

The predictor uses its **own predictions** (not ground truth) for subsequent steps.
This prevents compounding errors at inference time.

### Total Loss
$$\mathcal{L}(\phi) = \mathcal{L}_{\text{tf}} + \mathcal{L}_{\text{ar}}$$

In [ ]:
# MULTI-STEP ROLLOUT VISUALIZATION
# Show how error accumulates in autoregressive vs teacher-forcing

def visualize_rollout_error():
    """Compare error accumulation in teacher-forcing vs autoregressive rollout."""
    
    T = 10  # prediction horizon
    D_latent = 64
    
    # Simulate ground truth trajectory
    gt_states = [torch.randn(D_latent)]
    for t in range(T):
        gt_states.append(gt_states[-1] + torch.randn(D_latent) * 0.1)
    gt_states = torch.stack(gt_states)  # [T+1, D]
    
    # Simulate predictor with some error
    pred_noise = 0.05  # prediction error per step
    
    # Teacher-forcing: always use GT input
    tf_errors = []
    for t in range(T):
        predicted = gt_states[t] + torch.randn(D_latent) * pred_noise  # predict from GT
        error = torch.mean(torch.abs(predicted - gt_states[t+1])).item()
        tf_errors.append(error)
    
    # Autoregressive: use own predictions
    ar_errors = []
    current = gt_states[0].clone()
    for t in range(T):
        predicted = current + torch.randn(D_latent) * pred_noise + (current - gt_states[t]) * 0.3
        error = torch.mean(torch.abs(predicted - gt_states[t+1])).item()
        ar_errors.append(error)
        current = predicted  # feed prediction back in!
    
    # Combined training
    combined_errors = []
    current_c = gt_states[0].clone()
    for t in range(T):
        # Mix GT and own predictions (simulates combined training effect)
        blend = min(t / T, 0.5)  # gradually rely more on own predictions
        input_state = (1 - blend) * gt_states[t] + blend * current_c
        predicted = input_state + torch.randn(D_latent) * pred_noise
        error = torch.mean(torch.abs(predicted - gt_states[t+1])).item()
        combined_errors.append(error)
        current_c = predicted
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Error over time
    ax = axes[0]
    steps = list(range(1, T+1))
    ax.plot(steps, tf_errors, 'g-o', linewidth=2, markersize=5, label='Teacher-Forcing (GT input)')
    ax.plot(steps, ar_errors, 'r-o', linewidth=2, markersize=5, label='Autoregressive (own predictions)')
    ax.plot(steps, combined_errors, 'b-o', linewidth=2, markersize=5, label='Combined (TF + AR training)')
    ax.set_xlabel('Prediction Step')
    ax.set_ylabel('L1 Error vs Ground Truth')
    ax.set_title('Error Accumulation Over Time')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Architecture diagram (text-based)
    ax = axes[1]
    ax.axis('off')
    
    diagram_text = """
    TEACHER-FORCING (training time):
    ┌─────┐   ┌─────┐   ┌─────┐   ┌─────┐
    │GT z₀│──►│GT z₁│──►│GT z₂│──►│GT z₃│  ← all GT
    └──┬──┘   └──┬──┘   └──┬──┘   └──┬──┘
       │         │         │         │
       ▼         ▼         ▼         ▼
    ┌──────┐ ┌──────┐ ┌──────┐ ┌──────┐
    │ẑ₁=P()│ │ẑ₂=P()│ │ẑ₃=P()│ │ẑ₄=P()│  ← predictions
    └──────┘ └──────┘ └──────┘ └──────┘
    
    AUTOREGRESSIVE (inference time):
    ┌─────┐   ┌──────┐   ┌──────┐   ┌──────┐
    │GT z₀│──►│ẑ₁=P()│──►│ẑ₂=P()│──►│ẑ₃=P()│
    └─────┘   └──┬───┘   └──┬───┘   └──┬───┘
                 │           │           │
              uses ẑ₁     uses ẑ₂     uses ẑ₃
              (error       (error      (errors
               grows!)      grows!)     compound!)
    """
    ax.text(0.05, 0.5, diagram_text, fontfamily='monospace', fontsize=9,
            verticalalignment='center', transform=ax.transAxes)
    ax.set_title('Why Autoregressive Training Matters')
    
    plt.tight_layout()
    plt.show()
    
    print("Without AR training: model is trained on perfect inputs but tested on noisy ones")
    print("The mismatch causes errors to compound exponentially at inference")
    print("AR training during training teaches the model to correct its own mistakes")

visualize_rollout_error()

In [ ]:
def demonstrate_ac_training_loop():
    """
    Minimal implementation of the V-JEPA 2-AC training losses.
    Source: refs/vjepa2/app/vjepa_droid/train.py, lines 403-470
    """
    # Simplified dimensions
    B, T, HW, D = 2, 8, 16, 64  # 2 videos, 8 frames, 4x4 spatial, 64-dim
    action_dim = 7
    tokens_per_frame = HW
    auto_steps = 2  # from config: how many AR rollout steps
    
    # Simulated data
    clips = torch.randn(B, 3, T, 64, 64)  # raw video (not used directly)
    actions = torch.randn(B, T - 1, action_dim)
    states = torch.randn(B, T, action_dim)
    
    # Step 1: Target encoder (frozen) — encode each frame independently
    # In real code: c.permute(0,2,1,3,4).flatten(0,1).unsqueeze(2).repeat(1,1,2,1,1)
    # This treats each frame as a 2-frame tubelet by repeating it
    h = torch.randn(B, T * tokens_per_frame, D)  # target representations
    h = F.layer_norm(h, (D,))  # normalize_reps=True
    
    # Simulated predictor (returns predictions)
    def fake_predictor(z, a, s):
        """Simulates AC predictor output."""
        T_in = z.shape[1] // tokens_per_frame
        # In reality: interleave actions/states, apply frame-causal attention
        # Here we just return shifted version as a simple proxy
        noise = torch.randn_like(z) * 0.1
        return z + noise  # simplified
    
    # =============================================
    # LOSS 1: Teacher-Forcing
    # =============================================
    # Input: frames 0..T-2 (ground truth from target encoder)
    # Target: frames 1..T-1
    z_input = h[:, :-tokens_per_frame]  # all frames except last
    z_tf = fake_predictor(z_input, actions, states[:, :-1])
    
    # Target: frames 1..T-1 from target encoder
    h_target = h[:, tokens_per_frame:]  # skip first frame
    
    # Crop to match sizes
    min_len = min(z_tf.shape[1], h_target.shape[1])
    jloss = torch.mean(torch.abs(z_tf[:, :min_len] - h_target[:, :min_len]))
    
    # =============================================
    # LOSS 2: Autoregressive Rollout
    # =============================================
    # Start: frames 0,1 from ground truth
    z_ar = torch.cat([
        h[:, :tokens_per_frame],      # frame 0 (GT)
        z_tf[:, :tokens_per_frame]     # frame 1 (predicted by TF)
    ], dim=1)
    
    # Rollout: use own predictions for subsequent frames
    for n in range(1, auto_steps):
        a_slice = actions[:, :n+1]
        s_slice = states[:, :n+1]
        z_next = fake_predictor(z_ar, a_slice, s_slice)
        z_next = z_next[:, -tokens_per_frame:]  # only last frame
        z_ar = torch.cat([z_ar, z_next], dim=1)
    
    z_ar_preds = z_ar[:, tokens_per_frame:]  # remove first frame
    h_ar_target = h[:, tokens_per_frame:z_ar_preds.shape[1]+tokens_per_frame]
    
    min_len = min(z_ar_preds.shape[1], h_ar_target.shape[1])
    sloss = torch.mean(torch.abs(z_ar_preds[:, :min_len] - h_ar_target[:, :min_len]))
    
    # =============================================
    # TOTAL LOSS
    # =============================================
    total_loss = jloss + sloss
    
    print("V-JEPA 2-AC Training Losses")
    print("=" * 40)
    print(f"Teacher-forcing loss (L_tf): {jloss.item():.4f}")
    print(f"  → Predictor gets ground-truth frames, predicts next frame")
    print(f"  → Trains accurate single-step predictions")
    print(f"")
    print(f"Autoregressive loss (L_ar):  {sloss.item():.4f}")
    print(f"  → Predictor uses OWN predictions for multi-step rollout")
    print(f"  → Prevents compounding errors at inference")
    print(f"  → auto_steps={auto_steps} (from config)")
    print(f"")
    print(f"Total loss:                  {total_loss.item():.4f}")
    print(f"  → L_total = L_tf + L_ar (equal weighting)")

demonstrate_ac_training_loop()

In [ ]:
# CEM PLANNING: Detailed energy landscape visualization

def visualize_cem_energy_landscape():
    """Visualize how CEM narrows its search in the action space."""
    
    torch.manual_seed(42)
    np.random.seed(42)
    
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    
    # Create a 2D energy landscape (simplified action space for visualization)
    x = np.linspace(-1, 1, 100)
    y = np.linspace(-1, 1, 100)
    X, Y = np.meshgrid(x, y)
    
    # Energy function: multiple local minima, one global minimum
    E = (0.5 * ((X - 0.3)**2 + (Y - 0.4)**2) + 
         0.3 * np.sin(5*X) * np.cos(5*Y) + 
         0.2 * np.exp(-((X+0.5)**2 + (Y-0.3)**2) / 0.1))
    
    # CEM iterations
    mu = np.array([0.0, 0.0])
    sigma = np.array([0.5, 0.5])
    n_candidates = 200
    n_elites = 20
    
    for iteration in range(5):
        ax = axes[iteration // 3][iteration % 3]
        
        # Draw energy landscape
        ax.contourf(X, Y, E, levels=20, cmap='YlOrRd', alpha=0.6)
        ax.contour(X, Y, E, levels=10, colors='gray', alpha=0.3, linewidths=0.5)
        
        # Sample candidates
        candidates = np.random.randn(n_candidates, 2) * sigma + mu
        candidates = np.clip(candidates, -1, 1)
        
        # Evaluate energy
        energies = (0.5 * ((candidates[:, 0] - 0.3)**2 + (candidates[:, 1] - 0.4)**2) + 
                    0.3 * np.sin(5*candidates[:, 0]) * np.cos(5*candidates[:, 1]) +
                    0.2 * np.exp(-((candidates[:, 0]+0.5)**2 + (candidates[:, 1]-0.3)**2) / 0.1))
        
        # Select elites
        elite_idx = np.argsort(energies)[:n_elites]
        elites = candidates[elite_idx]
        
        # Plot candidates and elites
        ax.scatter(candidates[:, 0], candidates[:, 1], c='blue', s=5, alpha=0.2, label='Candidates')
        ax.scatter(elites[:, 0], elites[:, 1], c='lime', s=30, edgecolors='black', 
                   linewidths=0.5, label='Elites', zorder=5)
        
        # Plot distribution
        from matplotlib.patches import Ellipse
        ell = Ellipse(xy=mu, width=2*sigma[0], height=2*sigma[1], 
                      fill=False, color='cyan', linewidth=2, linestyle='--')
        ax.add_patch(ell)
        
        # Mark goal
        ax.plot(0.3, 0.4, 'r*', markersize=15, zorder=10, label='Goal')
        
        ax.set_xlim(-1, 1)
        ax.set_ylim(-1, 1)
        ax.set_title(f'CEM Iteration {iteration}\nμ=({mu[0]:.2f},{mu[1]:.2f}), σ=({sigma[0]:.2f},{sigma[1]:.2f})')
        ax.set_aspect('equal')
        if iteration == 0:
            ax.legend(fontsize=7, loc='lower left')
        
        # Update distribution from elites
        mu = elites.mean(axis=0)
        sigma = elites.std(axis=0) + 0.01
    
    # Final plot: convergence summary
    ax = axes[1][2]
    ax.axis('off')
    summary = """
    CEM Planning Summary:
    ─────────────────────
    1. Start: wide Gaussian over action space
    2. Sample 800 candidate action sequences
    3. Evaluate each via world model:
       E(a) = ‖P(a, s, z) - z_goal‖₁
    4. Keep top 10% (80 elites)
    5. Fit new Gaussian to elites
    6. Repeat 5 iterations
    7. Execute ONLY first action
    8. Observe new state → REPLAN
    
    V-JEPA 2-AC parameters:
    • 800 candidates
    • 5 CEM iterations
    • action_bound = 0.075 (L1-ball)
    • Planning horizon = 5 steps
    • Replanning every step
    • Total: ~16 sec per action
    """
    ax.text(0.05, 0.95, summary, fontfamily='monospace', fontsize=9,
            verticalalignment='top', transform=ax.transAxes,
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    
    plt.tight_layout()
    plt.show()

visualize_cem_energy_landscape()

## 6. Planning with CEM in Latent Space

At inference, V-JEPA 2-AC plans by **searching for optimal action sequences in latent space**.

### Energy Function
$$\mathcal{E}(\hat{a}_{1:T}; z_k, s_k, z_g) = \| P(\hat{a}_{1:T}; s_k, z_k) - z_g \|_1$$

where $z_g = E(x_g)$ is the encoded goal image.

### Cross-Entropy Method (CEM)
1. Initialize Gaussian: $\hat{a}_i \sim \mathcal{N}(\mu_i, \sigma_i^2)$
2. Sample 800 candidate action sequences
3. Evaluate energy for each candidate
4. Select top-k elites (lowest energy)
5. Update $\mu, \sigma$ from elite statistics
6. Repeat for several iterations
7. Execute ONLY the first action $a_1^*$, then **replan**

In [ ]:
def cem_planning_demo():
    """
    Demonstrate CEM planning in V-JEPA 2-AC's latent space.
    This is how the robot decides what action to take.
    """
    # Planning parameters (from the paper)
    action_dim = 7
    planning_horizon = 5     # plan T steps ahead
    n_candidates = 800       # sample 800 action sequences
    n_elites = 80            # keep top 80
    n_iterations = 5         # CEM iterations
    action_bound = 0.075     # L1-ball radius (~13cm max displacement)
    
    # Simulated latent space
    latent_dim = 64
    z_current = torch.randn(latent_dim)   # current state encoding
    z_goal = torch.randn(latent_dim)      # goal state encoding
    
    # Simulated world model (in reality: the AC predictor)
    def world_model(z, actions):
        """Predict future state given current state + actions."""
        z_pred = z.clone()
        for t in range(actions.shape[1]):
            # Simplified: real model does transformer forward pass
            z_pred = z_pred + actions[:, t, :latent_dim % action_dim].sum(-1, keepdim=True).expand_as(z_pred) * 0.1
        return z_pred
    
    # Initialize CEM distribution
    mu = torch.zeros(planning_horizon, action_dim)
    sigma = torch.ones(planning_horizon, action_dim) * 0.02
    
    print("CEM Planning in V-JEPA 2-AC Latent Space")
    print("=" * 50)
    
    for iteration in range(n_iterations):
        # Step 1: Sample candidate action sequences
        noise = torch.randn(n_candidates, planning_horizon, action_dim)
        candidates = mu.unsqueeze(0) + sigma.unsqueeze(0) * noise
        
        # Clip to action bounds
        candidates = candidates.clamp(-action_bound, action_bound)
        
        # Step 2: Evaluate energy for each candidate
        z_curr_batch = z_current.unsqueeze(0).expand(n_candidates, -1)
        z_predicted = world_model(z_curr_batch, candidates)
        
        # Energy = L1 distance to goal in latent space
        energies = torch.mean(torch.abs(z_predicted - z_goal.unsqueeze(0)), dim=-1)
        
        # Step 3: Select elites
        elite_indices = energies.argsort()[:n_elites]
        elites = candidates[elite_indices]
        
        # Step 4: Update distribution
        mu = elites.mean(dim=0)
        sigma = elites.std(dim=0) + 1e-6
        
        print(f"  Iter {iteration}: mean energy = {energies[elite_indices].mean():.4f}, "
              f"best = {energies[elite_indices[0]]:.4f}")
    
    # Execute only the FIRST action
    best_action_sequence = mu
    action_to_execute = best_action_sequence[0]
    
    print(f"")
    print(f"Best action sequence found (planning horizon = {planning_horizon}):")
    for t in range(planning_horizon):
        print(f"  t={t}: {best_action_sequence[t].tolist()[:3]}... (showing first 3 of 7 dims)")
    print(f"")
    print(f"Execute ONLY a[0] = {action_to_execute.tolist()[:3]}..., then REPLAN.")
    print(f"")
    print(f"Planning takes ~16 seconds (800 candidates, V-JEPA 2-AC).")
    print(f"Compare: Cosmos (pixel-space) takes ~4 minutes with only 80 candidates.")

cem_planning_demo()

## 7. V-JEPA 2-AC Results Summary

| Metric | V-JEPA 2-AC | Octo (VLA) | Cosmos (Video Gen) |
|--------|-------------|------------|-------------------|
| Pick-and-place success | **65-80%** | 15% | ~50% |
| Robot training data | 62 hours | 800K episodes | Millions |
| Planning time per action | **16 sec** | 10ms | 4 min |
| Task-specific training | **None (zero-shot)** | Required | Required |

**Key takeaway:** V-JEPA 2-AC achieves strong performance with:
- Zero task-specific training (zero-shot generalization)
- Only 62 hours of unlabeled robot video for post-training
- 15x faster planning than pixel-space alternatives

The world model pre-trained on internet video transfers physical understanding to robotics.

---

**Next notebook:** How JEPA integrates with VLA models (JEPA-VLA and VLA-JEPA papers).